In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from retinaface import RetinaFace
import numpy as np
import matplotlib.pyplot as plt
import csv
import cv2
from scipy.spatial.distance import euclidean

In [ ]:
# Input data
img_dir = "data/rd_images"
img_paths = [os.path.join(img_dir, item) for item in os.listdir(img_dir) if os.path.isfile(os.path.join(img_dir, item))]

bnw_dir = "data_aug/preprocessed/bnw"
bnw_paths = [os.path.join(bnw_dir, item) for item in os.listdir(bnw_dir) if os.path.isfile(os.path.join(bnw_dir, item))]

color_dir = "data_aug/preprocessed/color"
color_paths = [os.path.join(color_dir, item) for item in os.listdir(color_dir) if os.path.isfile(os.path.join(color_dir, item))]

# Output folders
bnw_sr = "data_aug/preprocessed/bnw_sr"
color_sr = "data_aug/preprocessed/color_sr"
padded_dir = "data_aug/preprocessed/padded_sr"

os.makedirs(bnw_sr, exist_ok=True)
os.makedirs(color_sr, exist_ok=True)
os.makedirs(padded_dir, exist_ok=True)

In [ ]:
# check number of subfolders in rd_images and also number of images in each subfolder
def check_subfolders_and_images(img_dir):
    class_and_images = {}
    subfolders = [f.path for f in os.scandir(img_dir) if f.is_dir()]
    # print(f"Number of subfolders: {len(subfolders)}")
    for folder in subfolders:
        images = [f.path for f in os.scandir(folder) if f.is_file()]
        # print(f"Number of images in {os.path.basename(folder)}: {len(images)}")
        class_and_images[os.path.basename(folder)] = len(images)
    return class_and_images

class_and_images = check_subfolders_and_images(img_dir)

In [ ]:
class_names = np.array(list(class_and_images.keys()))
images_count = np.array(list(class_and_images.values()))
print(f"Number of subfolders: {len(class_and_images)}")
print(f"Total number of images: {np.sum(images_count)}")
number_count = [np.sum(images_count == i) for i in range(1,8)]
print(class_names)
print(images_count)

print(np.sum(number_count))
print(number_count)

# a plot of the number distribution and mean and median on the plot
plt.figure(figsize=(12, 5))
plt.bar(range(1, 8), number_count)
plt.xticks(range(1, 8))
plt.xlabel("Number of images")
plt.ylabel("Number of classes")
plt.title("Distribution of number of images per disease class")

plt.show()

In [ ]:
# function to resize and pad images for better facial detection
def resize_and_pad(img, target_size=(512, 512), fill_color=(0, 0, 0)):
    old_size = img.size  # (width, height)
    ratio = min(target_size[0] / old_size[0], target_size[1] / old_size[1])
    new_size = (int(old_size[0] * ratio), int(old_size[1] * ratio))
    
    img = img.resize(new_size, Image.LANCZOS)
    
    new_img = Image.new("RGB", target_size, fill_color)
    paste_position = ((target_size[0] - new_size[0]) // 2, (target_size[1] - new_size[1]) // 2)
    new_img.paste(img, paste_position)
    
    return new_img


for path in tqdm(img_paths):
    try:
        img = Image.open(path).convert("RGB")
        resized_padded_img = resize_and_pad(img, target_size=(512, 512))
        new_path = os.path.join(resized_dir, os.path.basename(path))
        resized_padded_img.save(new_path)
    except Exception as e:
        print(f"Resize error for {path}: {e}")


## High-resolution

In [ ]:
# Requires: Real-ESRGAN setup in /path/to/Real-ESRGAN
realesrgan_path = "Real-ESRGAN"

# for black-and-white images
for path in tqdm(bnw_paths):
    os.system(f"python {realesrgan_path}/inference_realesrgan.py -i {path} -o {bnw_sr} -n RealESRGAN_x4plus --outscale 3.5 --face_enhance")

# for colored images
for path in tqdm(color_paths):
    os.system(f"python {realesrgan_path}/inference_realesrgan.py -i {path} -o {color_sr} -n RealESRGAN_x4plus --outscale 3.5 --face_enhance")


In [ ]:
input_dir = "data_aug/preprocessed/color"
output_dir = color_sr

# File extensions to consider (e.g., .jpg, .png)
valid_exts = ['.png', '.jpg', '.jpeg']

# List original images (excluding hidden files)
input_files = [
    os.path.splitext(f)[0]
    for f in os.listdir(input_dir)
    if os.path.isfile(os.path.join(input_dir, f)) and os.path.splitext(f)[1].lower() in valid_exts
]

# List output images (remove "_out" suffix)
output_files = [
    os.path.splitext(f)[0].replace('_out', '')
    for f in os.listdir(output_dir)
    if os.path.isfile(os.path.join(output_dir, f)) and os.path.splitext(f)[1].lower() in valid_exts and '_out' in f
]

# Compare and report missing ones
missing_outputs = set(input_files) - set(output_files)

print(f"\nTotal original images: {len(input_files)}")
print(f"Total enhanced images: {len(output_files)}")
print(f"Missing enhanced images: {len(missing_outputs)}")

if missing_outputs:
    print("\nFiles missing enhancement:")
    for fname in sorted(missing_outputs):
        print(fname)
else:
    print("\n All original images have enhanced outputs.")


## Colorization

In [ ]:
# ddcolor to colorize black and white images
from modelscope.hub.snapshot_download import snapshot_download

model_dir = snapshot_download('damo/cv_ddcolor_image-colorization', cache_dir='./modelscope')
print('model assets saved to %s' % model_dir)

os.system(
    f'python DDColor/infer.py --model_path DDColor/modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt '
    f'--input "{bnw_sr}" --output "{color_sr}"'
)

# now all colorized and high-res images are in color_sr


## Landmark detection

In [ ]:
# padded images to retinaFace to better recognize faces on the images
def pad_image(img, pad_ratio=0.5):
    h, w = img.shape[:2]
    pad_h, pad_w = int(h * pad_ratio), int(w * pad_ratio)
    padded_img = cv2.copyMakeBorder(img, pad_h, pad_h, pad_w, pad_w, borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0])
    return padded_img

color_sr_paths = [os.path.join(color_sr, item) for item in os.listdir(color_sr) if os.path.isfile(os.path.join(color_sr, item))]
for path in tqdm(color_sr_paths):
    img = cv2.imread(path)
    padded_img = pad_image(img, pad_ratio=0.5)
    output_dir = padded_dir
    cv2.imwrite(os.path.join(output_dir, os.path.basename(path)), padded_img)

In [ ]:
# generate metadata including landmarks for each faces into a csv file
padded_paths = [os.path.join(padded_dir, item) for item in os.listdir(padded_dir) if os.path.isfile(os.path.join(padded_dir, item))]
csv_file = "data_aug/face_info.csv"
fieldnames = [
    "image_path", "face_id", "score",
    "facial_area_x1", "facial_area_y1", "facial_area_x2", "facial_area_y2",
    "right_eye_x", "right_eye_y",
    "left_eye_x", "left_eye_y",
    "nose_x", "nose_y",
    "mouth_right_x", "mouth_right_y",
    "mouth_left_x", "mouth_left_y"
]
# Open CSV once and append rows
with open(csv_file, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()

    padded_paths = [
        os.path.join(padded_dir, item)
        for item in os.listdir(padded_dir)
        if os.path.isfile(os.path.join(padded_dir, item))
    ]

    for path in tqdm(padded_paths):
        face_info = RetinaFace.detect_faces(path, threshold=0.8)

        if isinstance(face_info, dict):
            for face_id, data in face_info.items():
                row = {
                    "image_name": os.path.basename(path),
                    "face_id": face_id,
                    "score": data.get("score", ""),
                    "facial_area_x1": data["facial_area"][0],
                    "facial_area_y1": data["facial_area"][1],
                    "facial_area_x2": data["facial_area"][2],
                    "facial_area_y2": data["facial_area"][3],
                    "right_eye_x": data["landmarks"]["right_eye"][0], # note: right eye is on the left side of the image
                    "right_eye_y": data["landmarks"]["right_eye"][1],
                    "left_eye_x": data["landmarks"]["left_eye"][0], # note: left eye is on the right side of the image
                    "left_eye_y": data["landmarks"]["left_eye"][1],
                    "nose_x": data["landmarks"]["nose"][0],
                    "nose_y": data["landmarks"]["nose"][1],
                    "mouth_right_x": data["landmarks"]["mouth_right"][0],
                    "mouth_right_y": data["landmarks"]["mouth_right"][1],
                    "mouth_left_x": data["landmarks"]["mouth_left"][0],
                    "mouth_left_y": data["landmarks"]["mouth_left"][1],
                }
                writer.writerow(row)
        else:
            print(f"Face detection failed for {path}")
            continue

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import euclidean

# Load dataset
df = pd.read_csv("data_aug/face_info.csv")
image_names = df['image_name']

# Define the column names for landmarks
landmark_names = [
    ('right_eye_x', 'right_eye_y'),
    ('left_eye_x', 'left_eye_y'),
    ('nose_x', 'nose_y'),
    ('mouth_right_x', 'mouth_right_y'),
    ('mouth_left_x', 'mouth_left_y')
]

# Function to compute normalized 5x5 distance matrix
def compute_normalized_landmark_matrix(row):
    x1, y1 = row['facial_area_x1'], row['facial_area_y1']
    x2, y2 = row['facial_area_x2'], row['facial_area_y2']
    w = x2 - x1
    h = y2 - y1

    if w == 0 or h == 0:
        return np.full((5, 5), np.nan)  # or handle differently

    # Normalize landmarks
    landmarks = np.array([
        [ (row[x] - x1) / w, (row[y] - y1) / h ]
        for x, y in landmark_names
    ])

    # Compute distances
    dist_matrix = np.zeros((5, 5))
    for i in range(5):
        for j in range(5):
            dist_matrix[i, j] = euclidean(landmarks[i], landmarks[j])

    return dist_matrix

# Apply the function to each row
distance_matrices = df.apply(compute_normalized_landmark_matrix, axis=1)

# Flatten each matrix into a 25-element row vector
flat_matrices = distance_matrices.apply(lambda mat: mat.flatten())

# Create a DataFrame from the flattened vectors
feature_columns = [f'dist_{i}_{j}' for i in range(5) for j in range(5)]
dist_df = pd.DataFrame(flat_matrices.tolist(), columns=feature_columns)

# Add image_name for reference
dist_df.insert(0, 'image_name', image_names)

# Save to CSV
dist_df.to_csv("data_aug/landmark_distances.csv", index=False)


In [ ]:
# change the image names to the orignial ones and put them into subfolders
import os
import shutil

main_folder = "data_aug/preprocessed/final"
os.makedirs(main_folder, exist_ok=True)
color_sr = "data_aug/preprocessed/color_sr"

folder_names = os.listdir("data/rd_images")

for folder_name in folder_names:
    os.makedirs(os.path.join(main_folder, folder_name), exist_ok=True)
    
# put images into subfolders
for img_name in os.listdir(color_sr):
    # chunk the image names
    class_name = img_name.split(".")[0]
    idx = img_name.split(".")[1].split("_")[0]
    shutil.copy(os.path.join(color_sr, img_name), os.path.join(main_folder, class_name, class_name + "." + idx + "." + img_name.split(".")[2]))
    

In [ ]:
# # remove subfolders in the main folder
# import os
# import shutil

# # Set your main folder path
# main_folder = 'data_aug/rd_images'

# # Loop through each subfolder
# for root, dirs, files in os.walk(main_folder, topdown=False):
#     if root == main_folder:
#         continue  # skip the main folder itself

#     for file in files:
#         # Full path to the source file
#         src = os.path.join(root, file)
#         # Full path to the destination (main folder)
#         dst = os.path.join(main_folder, file)

#         # Rename the file if a file with the same name already exists
#         base, ext = os.path.splitext(file)
#         counter = 1
#         while os.path.exists(dst):
#             dst = os.path.join(main_folder, f"{base}_{counter}{ext}")
#             counter += 1

#         shutil.move(src, dst)

#     # After moving files, remove the empty subfolder
#     os.rmdir(root)

# print("All images moved to the main folder and subfolders deleted.")
